In [1]:
# pipenv install ipykernel pandas ipywidgets plotly scikit-learn gradio optuna nbformat matplotlib pingouin 

# EDA
import pandas as pd
import plotly.express as px

pd.set_option('display.float_format', lambda x: '%.2f' %  x)

# ML
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer


import optuna

# EDA

## Loading the dataset

In [2]:
df_clients = pd.read_csv('./datasets/client_dataset.csv')
df_clients.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   economic_activity    500 non-null    str    
 1   monthly_revenue      500 non-null    float64
 2   number_of_employees  500 non-null    int64  
 3   location             500 non-null    str    
 4   age                  500 non-null    int64  
 5   innovation           500 non-null    int64  
dtypes: float64(1), int64(3), str(2)
memory usage: 23.6 KB


In [3]:
df_clients.head(10)

,economic_activity,monthly_revenue,number_of_employees,location,age,innovation
0,Commerce,713109.95,12,Rio de Janeiro,6,1
1,Commerce,790714.38,9,São Paulo,15,0
2,Commerce,1197239.33,17,São Paulo,4,9
3,Industry,449185.78,15,São Paulo,6,0
4,Agribusiness,1006373.16,15,São Paulo,15,8
5,Services,1629562.41,16,Rio de Janeiro,11,4
6,Services,771179.95,13,Vitória,0,1
7,Services,707837.61,16,São Paulo,10,6
8,Commerce,888983.66,17,Belo Horizonte,10,1
9,Industry,1098512.64,13,Rio de Janeiro,9,3


## Exploring the data

In [4]:
innovation_percentual = df_clients.value_counts('innovation') / len(df_clients) * 100

px.bar(innovation_percentual, color=innovation_percentual.index)

### ANOVA Test
Determine whether there are significant differences in the mean monthly revenue across different levels of innovation

Assumptions:
- Independent observations
- The dependent variable is continuous
- It follows a normal distribution
- Homogeneity of variances
- Samples are of equal size



In [7]:
# Check whether the variances (monthly revenue) between the groups (innovation) are homogeneous
# Apply the Bartlett Test
# H0 - Variances are the same
# H1 - Variances are not the same

from scipy.stats import bartlett

# Splitting the revenue data in groups based on the ‘innovation’ column
grouped_data = [df_clients.monthly_revenue[df_clients['innovation'] == group] for group in df_clients['innovation'].unique()]

# Running the test
bartlett_test_statistic, bartlett_p_value = bartlett(*grouped_data)

# Showing the results
print(f"Bartlett Test Stats: {bartlett_test_statistic}")
print(f"Bartlett Test P-Value: {bartlett_p_value}")

Bartlett Test Stats: 10.901203117231173
Bartlett Test P-Value: 0.28254182954905804


In [8]:
# Running the Shapiro-Wilk Test
# Verifying whether the data follows a normal distribution
# H0 - Follows a normal distribution
# H1 - Don't follows a normal distribution

from scipy.stats import shapiro

# Running the test
shapiro_test_statistic, shapiro_p_value = shapiro(df_clients['monthly_revenue'])

# Showing the results
print(f"SW Test Stats: {shapiro_test_statistic}")
print(f"SW Test P-Value: {shapiro_p_value}")

SW Test Stats: 0.9959857602472711
SW Test P-Value: 0.23513451034389005


In [11]:
# Use Welch's ANOVA because the sample sizes are different
# H0: There is no significant difference between the group means
# H1: There is at least one significant difference between the group means

from pingouin import welch_anova

aov = welch_anova(dv='monthly_revenue', between='innovation', data=df_clients)

# Showing the results
print(f"Welch's ANOVA Test Stats: {aov.loc[0, 'F']}")
print(f"Welch's ANOVA P-Value: {aov.loc[0, 'p_unc']}")

Welch's ANOVA Test Stats: 1.1269836194061693
Welch's ANOVA P-Value: 0.3452621127391271


# Model Training

In [14]:
# Select the columns for clusterization
X = df_clients.copy()

# Distinguishing between categorical, ordinal and numerical features
numeric_features = ['monthly_revenue', 'number_of_employees', 'age']
categorical_features = ['location', 'economic_activity']
ordinal_features = ['innovation']

# Applying the Transformer
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()
ordinal_transformer = OrdinalEncoder()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('ord', ordinal_transformer, ordinal_features)
    ]
)

X_transformed = preprocessor.fit_transform(X)
X_transformed

array([[-0.74634498, -0.54179191, -1.10058849, ...,  0.        ,
         0.        ,  1.        ],
       [-0.56165548, -1.5035527 ,  1.94344851, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.40582654,  1.06114274, -1.77704115, ...,  0.        ,
         0.        ,  9.        ],
       ...,
       [ 2.8196246 , -1.18296577,  0.25231684, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.03321411, -0.54179191, -1.43881482, ...,  0.        ,
         0.        ,  3.        ],
       [-2.03011486, -0.22120498, -1.77704115, ...,  1.        ,
         0.        ,  9.        ]], shape=(500, 12))

In [16]:
# Optuna for hyperparameters optimization
def kmeans_objective(trial):
    # Defining the hyperparameters to be tuned
    n_clusters = trial.suggest_int('n_clusters', 3, 10)
    distance_metric = trial.suggest_categorical('distance_metric', ['euclidean', 'minkowski'])

    # Creating the model
    kmeans_model = KMeans(n_clusters=n_clusters, random_state=42)

    kmeans_model.fit(X_transformed)

    # Calculating the Silhouette Score
    distances = pairwise_distances(X_transformed, metric=distance_metric)
    silhouette_avg = silhouette_score(distances, kmeans_model.labels_)

    return silhouette_avg

In [18]:
search_space = {'n_clusters': [3,4,5,6,7,8,9,10], 'distance_metric': ['euclidean', 'minkowski']}
sampler = optuna.samplers.GridSampler(search_space=search_space)

kmeans_study = optuna.create_study(direction='maximize', sampler=sampler)

kmeans_study.optimize(kmeans_objective, n_trials=16)

[I 2026-06-14 13:03:34,346] A new study created in memory with name: no-name-3e2e6074-ea4d-4a17-8667-dfc3b878f8df
[I 2026-06-14 13:03:34,396] Trial 0 finished with value: 0.3887134965445921 and parameters: {'n_clusters': 4, 'distance_metric': 'euclidean'}. Best is trial 0 with value: 0.3887134965445921.
[I 2026-06-14 13:03:34,481] Trial 1 finished with value: 0.14058096932077915 and parameters: {'n_clusters': 9, 'distance_metric': 'euclidean'}. Best is trial 0 with value: 0.3887134965445921.
[I 2026-06-14 13:03:34,500] Trial 2 finished with value: 0.44454582909990875 and parameters: {'n_clusters': 3, 'distance_metric': 'minkowski'}. Best is trial 2 with value: 0.44454582909990875.
[I 2026-06-14 13:03:34,562] Trial 3 finished with value: 0.3887134965445921 and parameters: {'n_clusters': 4, 'distance_metric': 'minkowski'}. Best is trial 2 with value: 0.44454582909990875.
[I 2026-06-14 13:03:34,587] Trial 4 finished with value: 0.15460498751026852 and parameters: {'n_clusters': 8, 'distan

In [20]:
best_params = kmeans_study.best_params

best_kmeans = KMeans(n_clusters=best_params['n_clusters'], random_state=51)
best_kmeans.fit(X_transformed)

distances = pairwise_distances(X_transformed, metric=best_params['distance_metric'])
silhouette_avg = silhouette_score(distances, best_kmeans.labels_)

print(f'K (Clusters amount): {best_params['n_clusters']}')
print(f'Distance Metric: {best_params['distance_metric']}')
print(f'Silhouett Score: {silhouette_avg}')

K (Clusters amount): 3
Distance Metric: euclidean
Silhouett Score: 0.4445458290999088


In [21]:
df_clients['cluster'] = best_kmeans.labels_

In [22]:
df_clients.head(10)

,economic_activity,monthly_revenue,number_of_employees,location,age,innovation,cluster
0,Commerce,713109.95,12,Rio de Janeiro,6,1,0
1,Commerce,790714.38,9,São Paulo,15,0,0
2,Commerce,1197239.33,17,São Paulo,4,9,1
3,Industry,449185.78,15,São Paulo,6,0,0
4,Agribusiness,1006373.16,15,São Paulo,15,8,1
5,Services,1629562.41,16,Rio de Janeiro,11,4,2
6,Services,771179.95,13,Vitória,0,1,0
7,Services,707837.61,16,São Paulo,10,6,1
8,Commerce,888983.66,17,Belo Horizonte,10,1,0
9,Industry,1098512.64,13,Rio de Janeiro,9,3,2


## Validating Results

In [23]:
# Crossing Age and Revenue, showing the clusters
px.scatter(df_clients, x='age', y='monthly_revenue', color='cluster')

In [24]:
# Crossing Innovation and Revenue, showing the clusters
px.scatter(df_clients, x='innovation', y='monthly_revenue', color='cluster')

## Saving the model and Pipeline

In [25]:
import joblib

# Saving the model
joblib.dump(best_kmeans, 'client_clusterization_model.pkl')

joblib.dump(preprocessor, 'client_clusterization_pipeline.pkl')

['client_clusterization_pipeline.pkl']

# Batch app with Gradio

In [26]:
import gradio as gr 

model = joblib.load('./client_clusterization_model.pkl')
preprocessor = joblib.load('./client_clusterization_pipeline.pkl')

def clustering(file):
    # loading the CSV on a dataframe
    df_companys = pd.read_csv(file.name)

    # Transforming the DF data
    X_transformed = preprocessor.fit_transform(df_companys)

    model.fit(X_transformed)

    df_companys['cluster'] = model.labels_
    df_companys.to_csv('./output/cluster.csv', index=False)

    return './output/cluster.csv'

In [27]:
app = gr.Interface(
    clustering,
    gr.File(file_types=['.csv']),
    "file"
)

app.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


e:\Programação\Rocketseat\ml_models_practice\07. K-Means\Clients Clusterization\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
